# TTS Chatterbox playground

This notebook wil contain all the experiments and documentations regarding the [Chatterbox](https://github.com/resemble-ai/chatterbox) model.

>NOTE: the notebook has been executed on a machine with the following specs:

In [5]:
!winfetch


 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
                                  
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll

Fazei@LAPTOP-O9V24FCP
---------------------
OS: Windows 11 Home [64-bit]
Host: LENOVO 82K2
Kernel: 10.0.26200.0
Motherboard: LENOVO LNVNB161216
Uptime: 1 hour 3 minutes
Packages: 26 (choco)
Shell: PowerShell v5.1.26100.7920
Resolution: 1536x864
Terminal: Visual Studio Code
CPU: AMD Ryzen 5 5600H with Radeon Graphics @ 3.294GHz
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
GPU: AMD Radeon(TM) Graphics
Memory: 11,7 GiB / 13,86 GiB (84%)
Disk (C:): 416 GiB / 475 GiB (87%)

    

## Setting up a venv (Virtual Environment)

> this notebook uses Python 3.10.0

Before we start, make sure to have a venv (virtual environment) to run this model

this can be achieved in the following ways:

- Using python which will include pip
    ```
    python -m venv .venv-name
    ```

- Using UV which is a package manager for Python written in Rust
    ```
    uv venv .venv-name
    ```

## Python & Packages

To get TTS Bark working, make sure that your virtual environment has installed all the requirements in [requirements.txt](./requirements.txt)

this can be achieved in the following ways:

- Using pip (package manager for Python)
    ```
    pip install -r requirements
    ```

- Using uv (package manager for Python written in Rust)
    ```
    uv pip install -r requirements.txt
    ```

# Code implementation

> NOTE: if you run into an issue regarding the Nonetype, check out https://github.com/resemble-ai/chatterbox/issues/198

In [ ]:
from chatterbox.tts import ChatterboxTTS
from pathlib import Path
import torchaudio as ta
import torch
import re
import textwrap

a_dir = "../outputs"
a_output = "chatterbox-output.wav"
input_file = "../tts.txt"
speaker_path = "./target/james/thinghost_2_james_64kb_short.wav"

def cuda_availability():
    return "cuda" if torch.cuda.is_available() else "cpu"

def clean_text(f):
    if Path(f).suffix != ".txt":
        raise ValueError("Oh no! the file needs to be a .txt!")
    with open(f, 'r') as f:
        return re.sub(r'["\'\n]', '', f.read())

# Once this hit production, change it into configurable classes
model = ChatterboxTTS.from_pretrained(device=cuda_availability())
raw_text = clean_text(input_file)
chunks = textwrap.wrap(raw_text, width=250)

all_wavs = []
for chunk in chunks:
    wav = model.generate(chunk,audio_prompt_path=speaker_path,exaggeration=0.35,cfg_weight=0.2)
    all_wavs.append(wav)

final_wav = torch.cat(all_wavs, dim=-1)

ta.save(f"{a_dir}/{a_output}", final_wav, model.sr)



Sampling:  32%|███▏      | 319/1000 [01:05<02:20,  4.85it/s]

Sampling:  38%|███▊      | 378/1000 [01:15<02:03,  5.02it/s]

Sampling:  38%|███▊      | 383/1000 [01:17<02:05,  4.93it/s]

Sampling:  36%|███▋      | 363/1000 [01:21<02:23,  4.43it/s]

Sampling:  36%|███▋      | 363/1000 [01:29<02:36,  4.07it/s]

Sampling:  38%|███▊      | 383/1000 [01:30<02:26,  4.22it/s]

Sampling:  34%|███▍      | 344/1000 [01:24<02:41,  4.06it/s]

Sampling:  42%|████▏     | 421/1000 [01:44<02:24,  4.01it/s]

Sampling:  38%|███▊      | 377/1000 [01:32<02:33,  4.07it/s]

Sampling:  40%|███▉      | 397/1000 [01:31<02:19,  4.33it/s]

Sampling:  34%|███▍      | 339/1000 [01:18<02:32,  4.34it/s]

Sampling:   7%|▋         | 71/1000 [00:15<03:25,  4.52it/s]


## Notes regarding implementation

- The result of this model wil end up in the [output directory](../outputs/)

- Make sure to have a text file that you can put in the generator! As far as testing has gone it should work with .txt files.

- Make sure to have a ```target``` directory where you can place the audio file of the person or character that you would like to mimic. So far I have only tested it with ```.wav``` files, but ```.mp3``` files should also work

> NOTE: If this ever hits production make sure to turn it into configurable classes

## Findings

- It seems that the chatterbox model works the best with chunks of text between 200-250 characters, more than that and it will start hallucinating.

- In comparison to the XTTS-v2 it seems that it hallucinates less than expected, but it still talks quite fast.

- Each iteration of chunk processing to audio does take significantly longer in comparison to the kokoro and XTTS-v2 model.

- The way the model reads the text does sound natural, but do note that it does feel like the model breathes alot in between sentences, could probably be fixed with some tweaks.
